# DiscoCH: near real time vs. retrospective
Validates and compares the two runs of a season, side by side:

- **NRT** (`DiscoCH_NRT.ipynb`): each date interpolated from the observations up to 5 days after it, stepwise min/max of the NRT series.
- **Retro** (`DiscoCH_Retro.ipynb`): each date interpolated from all observations, stepwise min/max of the retrospective series.
- optionally **Retro (NRT min/max)** (`DiscoCH_Retro_NRTminmax.ipynb`): all-data interpolation with the NRT min/max — set `include_nrtminmax = True`.

Contents: the point validation and opportunistic-mapping recall tables, the NRT / Retro / 2018 time series and their difference, the per-region comparison, and a single-pixel interpolation example. Run the pipelines first. Outputs: `output/DiscoCH/<year>/nrt_vs_retro/`.

In [ ]:
# User defined parameters
# Paths are machine-specific and come from config.yaml in the project root
# (copy config.example.yaml to config.yaml and set the paths for your machine).
import calendar
from datetime import date
from src.disco_ch_operational.config import load_config

cfg = load_config()

year = 2026                    # season of both runs (the validation data below are from 2026)
threshold = 4000               # stored value (0-10000): 4000 = 40 % probability (validation, comparison season)
threshold_nrt = 4000           # time series plots: NRT discolored at >= 40 %
threshold_retro = 2000         # time series plots: Retro discolored at >= 20 %
include_nrtminmax = False      # also show the Retro run with the NRT min/max (DiscoCH_Retro_NRTminmax)

# Earlier season to compare with (Bloom et al. 2026 mosaics)
compare_year = 2018
mosaics_compare = cfg.require("mosaics_2018")
compare_pattern = "Proba_mosaic_*.tif"

In [ ]:
# Libraries
import os
import pandas as pd
import matplotlib.pyplot as plt

from src.disco_ch_operational.config import run_dirs
from src.disco_ch_operational.QAQC import optimized_point_table, optimized_recall_table, subsample_key
from src.disco_ch_operational.compare import plot_year_comparison, rc_params, BIOGEO_NAMES_EN, BIOGEO_COLORS
from src.disco_ch_operational.visualize_interpolation import plot_pixel

# Standard light theme for plots, white figures
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

districts = cfg.require("districts")            # political districts (cantons: field ktnr)
biogeo_regions = cfg.require("biogeo_regions")  # biogeographic regions (field DERegionNa)

# Folders of the runs and of this comparison
nrt_dirs = run_dirs(cfg["output_dir"], year, "nrt")
retro_dirs = run_dirs(cfg["output_dir"], year, "retro")
nrtmm_dirs = run_dirs(cfg["output_dir"], year, "retro_nrtminmax")
out = run_dirs(cfg["output_dir"], year, "nrt_vs_retro")

# The maps compared, and their names in tables and plots
maps = {"NRT": nrt_dirs["masked"], "Retro": retro_dirs["masked"]}
if include_nrtminmax:
    maps["Retro (NRT min/max)"] = nrtmm_dirs["masked"]
for name, path in maps.items():
    print(f"{name:20s} {path} ({'found' if os.path.isdir(path) else 'MISSING: run its notebook first'})")

## Validation
### Point validation
The labelled stratified random sample (`point_validation_dir` in config.yaml: `labeling_points.gpkg`, `sample_key.csv`, `strata_summary.csv`). Each map's probability is read at the sample points; the sample design (strata from the NRT maps, their pixel counts N_h) stays the same, so the area-weighted estimates are unbiased for every map (Stehman 2014). Outputs in `output/DiscoCH/<year>/nrt_vs_retro/validation/`: `table1_point_validation.csv`, `table_S1_point_validation_all_thresholds.csv` and `table2_recall_<date>.csv`.

#### Table 1: point validation at the optimized threshold
Per map, label confidence and date (all dates combined, then each date): the threshold (0.05 steps) with the highest area-weighted F1, and at it the area-weighted confusion matrix (TP, FP, FN, TN as shares of the area, summing to 1), precision, recall and F1, each ± 1.96 SE. **Table S1** (saved only) gives the same for every threshold 0.05–0.95, the optimized one marked with *.

In [ ]:
qa_dir = cfg.require("point_validation_dir")
labels = os.path.join(qa_dir, "labeling_points.gpkg")   # one layer per labelled date

# Exactly 100 points per date and probability class (0.2 wide): a fixed-seed random draw of the
# labelled points in the mask (classes with fewer keep all and are listed). Saved in validation/.
key100 = subsample_key(os.path.join(qa_dir, "sample_key.csv"), labels, n_per_class=100, seed=2026,
                       outfile=os.path.join(out["validation"], "sample_key_n100.csv"))

# Table 1 (shown) and Table S1, every threshold (saved only), in validation/
opt_long, table1 = optimized_point_table(
    maps,
    labels=labels,
    key=key100,
    summary=os.path.join(qa_dir, "strata_summary.csv"),
    min_confidence=(1, 2),             # All (>= 1) and Moderate to high (>= 2)
    outdir=out["validation"],
)
table1.style.hide(axis="index")


### Opportunistic mapping recall
Discolored polygons mapped from Planet imagery of 14 August (`validation_30km_gpkg`), against each map of that date. Without mapped non-discolored areas only recall can be computed.

#### Table 2: opportunistic mapping recall per canton at the optimized threshold
Pixel recall (share of the mapped pixels ≥ the threshold) of each map at its own optimized threshold from Table 1 (all dates, confidence moderate to high, ≥ 2). All of Switzerland on top, then the cantons from best to worst near-real-time recall.

In [ ]:
# Each map's optimized threshold of all dates, confidence >= 2 (Table 1)
opt_sel = opt_long[opt_long["optimized"] & (opt_long["date"] == "All") & (opt_long["min_confidence"] == 2)]
opt_thresholds = dict(zip(opt_sel["map"], opt_sel["threshold"]))
print("Thresholds:", {k: float(v) for k, v in opt_thresholds.items()})

recall2, table2 = optimized_recall_table(
    {k: maps[k] for k in ("NRT", "Retro")},   # NRT first: the table is sorted by its recall
    opt_thresholds,
    polygons=cfg.require("validation_30km_gpkg"),
    raster_name=f"{year}-08-14",   # the map of the Planet mapping date
    regions=districts, region_col="ktnr",
    names={"NRT": "Near-Real-Time", "Retro": "Retrospective"},
    outdir=out["validation"],      # table2_recall_<date>.csv
)
table2.style.hide(axis="index")

## Time Series
(A) Switzerland and (B) one panel per biogeographic region, NRT and Retro each at their own threshold (`threshold_nrt`, `threshold_retro`), the comparison season at `threshold`.

In [ ]:
# One A4-portrait figure, no title:
# (A) Switzerland: NRT solid grey, Retro solid black, the comparison season dashed black;
# (B) one panel per biogeographic region in its colour, the run by line style
#     (NRT dotted, Retro solid, comparison season dashed).
# Every dataset is counted per region here, so this takes a while.
nrt_label, retro_label = f"{year} (Near-Real-Time)", f"{year} (Retrospective)"
nrtmm_label, compare_label = f"{year} (Retrospective, NRT min/max)", f"{compare_year} (Retrospective)"
dataset_colors = {nrt_label: "0.6", retro_label: "black", nrtmm_label: "tab:blue", compare_label: "black"}
dataset_styles = {nrt_label: "-", retro_label: "-", nrtmm_label: "-.", compare_label: "--"}   # panel A
region_styles = {nrt_label: ":", retro_label: "-", nrtmm_label: "-.", compare_label: "--"}    # panel B

# Each run counted with its own threshold; datasets without one use `threshold`
datasets = {
    nrt_label: dict(mosaic_dir=nrt_dirs["masked"], year=year, threshold=threshold_nrt),
    retro_label: dict(mosaic_dir=retro_dirs["masked"], year=year, threshold=threshold_retro),
}
if include_nrtminmax:
    datasets[nrtmm_label] = dict(mosaic_dir=nrtmm_dirs["masked"], year=year)
datasets[compare_label] = dict(mosaic_dir=mosaics_compare, pattern=compare_pattern, year=compare_year)

comparison = plot_year_comparison(
    datasets,
    zones=biogeo_regions, zone_field="DERegionNa",   # regions = All + every DERegionNa
    region_names=BIOGEO_NAMES_EN,                   # German -> English names
    region_colors=BIOGEO_COLORS,                    # colour per region (and panel order)
    layout="summary",                               # A4 portrait (figsize default)
    linestyles=region_styles,
    national_colors=dataset_colors, national_linestyles=dataset_styles,
    title="",                                       # no figure title
    threshold=threshold,
    output_dir=out["statistics"],
    output_name=f"discoloration_{year}_nrt_vs_retro",
    max_workers=8,
)

In [ ]:
# Table: Switzerland per date -- share of the beech forest discolored (%) and its area (ha),
# for each dataset at its threshold; rows by month/day (the 5-day dates fall on the same
# days in both seasons). Saved as statistics/table_discoloration_<year>_nrt_vs_retro.csv.
nat = comparison[comparison.zone == "All"].copy()
nat["Date"] = pd.to_datetime(nat["date"]).dt.strftime("%m-%d")
order = list(datasets)
wide = nat.pivot(index="Date", columns="dataset", values=["proportion", "discolored_ha"])
thr_of = nat.groupby("dataset")["threshold"].first()

ts_table = pd.DataFrame(index=wide.index)
for lab in order:
    t = f" (t = {thr_of[lab] / 10000:.2f})" if pd.notna(thr_of.get(lab)) else ""
    ts_table[f"{lab}{t} %"] = wide[("proportion", lab)] * 100
    ts_table[f"{lab}{t} ha"] = wide[("discolored_ha", lab)]
ts_table = ts_table.sort_index().reset_index()
ts_table["Date"] = pd.to_datetime("2000-" + ts_table["Date"]).dt.strftime("%d %b")
ts_table.to_csv(os.path.join(out["statistics"], f"table_discoloration_{year}_nrt_vs_retro.csv"),
                index=False, encoding="utf-8-sig")
ts_table.style.hide(axis="index").format(
    {c: "{:.2f}" for c in ts_table.columns if c.endswith(" %")}
    | {c: "{:,.0f}" for c in ts_table.columns if c.endswith(" ha")}, na_rep="")


In [ ]:
# Difference per date: retrospective minus near real time (proportion of beech forest discolored)
total = comparison[comparison.zone == "All"].pivot(index="date", columns="dataset", values="proportion")
diff = (total[retro_label] - total[nrt_label]).dropna().rename("retro - nrt")
diff.to_csv(os.path.join(out["statistics"], f"discoloration_{year}_retro_minus_nrt.csv"))
print(diff.round(4).to_string())

## Interpolation Example
One pixel's observations and RBF interpolation: the comparison season (retrospective) and this season, near real time and retrospective.

In [ ]:
from datetime import date
from IPython.display import display

force_directory = cfg["force_dir"]
force_directory_compare = cfg.require("force_2018_dir")
as_of = min(date.today(), date(year, 12, 31))   # the near-real-time curve as at the end of the season

tile, vi = "X0056_Y0061", "EVI"
row, col = 2425, 2156  # pixel within the tile (None, None = pick one with enough observations)

# A4 portrait width, legends under each panel
fig, (ax_cmp, ax_year) = plt.subplots(2, 1, figsize=(8.27, 10.5))
plot_pixel(
    os.path.join(force_directory_compare, tile, f"{compare_year}-{compare_year}_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30, 50), rbf_cutoff=0.95, int_day=5,
    lag_days=None,  # retrospective ensemble only
    despike=True, above_noise=3.0, below_noise=1.0,
    title=f"{compare_year}, {tile} {vi}", ax=ax_cmp, component_data="retrospective",
    font_scale=0.85, legend="below",
)
plot_pixel(
    os.path.join(force_directory, tile, f"{year}0101-{year}1231_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30), rbf_cutoff=0.95, int_day=5,
    lag_days=5, as_of=as_of,           # near real time as in DiscoCH_NRT, with the retrospective curve
    component_data="nrt",              # kernel curves near-real-time ("retrospective" = all data)
    despike=False,
    title=f"{year}, {tile} {vi}", ax=ax_year,
    font_scale=0.85, legend="below",
)
# print_effective_bandwidth=True prints how much smoothing each date really got
os.makedirs(out["figures"], exist_ok=True)
fig.savefig(os.path.join(out["figures"], f"interpolation_{tile}_{vi}_r{row}_c{col}.png"), dpi=200, bbox_inches="tight")
display(fig)     # shown exactly once (plt.show() + the notebook's own flush can show it twice)
plt.close(fig)